## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## 2026: NEET status by poverty status

In [ ]:
row_values = {"NEET": 1, "Not NEET": 0}  # rows: neet = 1 is NEET, neet = 0 is not NEET
column_values = {"Poor": 1, "Not poor": 0}  # columns: poor = 1 is poor, poor = 0 is not poor

year26 = df[df["year"] == 2026]  # keep only the 2026 survey
known = year26.dropna(subset=["neet", "poor"])  # leave out people with neet or poor missing
n_left_out = len(year26) - len(known)  # how many were left out

percent = {}  # weighted percent, one row per NEET status
people = {}  # unweighted respondents, same layout
for row_label, neet_value in row_values.items():  # loop over NEET and Not NEET
    group = known[known["neet"] == neet_value]  # people in this row
    total_weight = group["asecwt"].sum()  # weighted number of people in this row
    percent[row_label] = {}  # start this row of the table
    people[row_label] = {}  # start this row of the counts
    for column_label, poor_value in column_values.items():  # loop over Poor and Not poor
        cell = group[group["poor"] == poor_value]  # people in this cell
        percent[row_label][column_label] = cell["asecwt"].sum() / total_weight * 100  # weighted percent of the row
        people[row_label][column_label] = len(cell)  # unweighted respondents in this cell

table = pd.DataFrame(percent).T  # dictionary -> table, one row per NEET status
people = pd.DataFrame(people).T  # same layout for the counts
off = (table.sum(axis=1) - 100).abs()  # distance from 100 in each row, before rounding

print("People in 2026 left out because neet or poor is missing (unweighted):", n_left_out)  # how many were left out
print()  # empty line
print("Table. NEET status by poverty status, 2026, % of people in each row (weighted with asecwt).")  # table title
print(table.to_string(float_format="{:.1f}".format))  # table, 1 decimal
print()  # empty line
print("Check: every row adds up to 100.0:", bool((off < 1e-9).all()))  # True or False
print()  # empty line
print("Unweighted number of respondents in each cell.")  # counts title
print(people.to_string())  # counts, same layout